# EDA — Marketing MMM Dataset

Owner: Ana Valderrama

**How to pass data (pick one):**

1. **Already cleaned via Streamlit** — use `data/processed/mmm_ready.csv` (created after Upload → Confirm).
2. **Raw CSV only** — set `RAW_CSV` below to your file path, then run the cell (pipeline runs automatically).
3. **No file yet** — leave `RAW_CSV = None`; a small synthetic sample is generated for demo only.

Run this notebook from `notebooks/` or the repo root — `config.yaml` is resolved automatically.

In [1]:
import sys
from pathlib import Path

_cwd = Path.cwd()
ROOT = _cwd if (_cwd / "config.yaml").exists() else _cwd.parent
sys.path.insert(0, str(ROOT))

from src.data_prep import generate_eda_report, project_root, run_pipeline

ROOT = project_root()  # always repo root (parent of src/)

import pandas as pd

# --- Option A: path to YOUR raw marketing CSV (or None) ---
# Example after Streamlit upload: ROOT / "data" / "raw" / "uploaded_dataset.csv"
# Example Conjura file: Path(r"C:/path/to/conjura_mmm_data.csv")
RAW_CSV = None

processed = ROOT / "data" / "processed" / "mmm_ready.csv"

if processed.exists():
    print("Using cleaned file:", processed)
    df = pd.read_csv(processed, parse_dates=["DATE_DAY"])
elif RAW_CSV is not None and Path(RAW_CSV).exists():
    print("Running pipeline on:", RAW_CSV)
    df = run_pipeline(raw_path=str(Path(RAW_CSV).resolve()))["ready_df"]
else:
    print("No processed/raw file found — using synthetic demo data")
    from datetime import datetime, timedelta

    dates = [datetime(2024, 1, 1) + timedelta(days=i) for i in range(30)]
    rows = []
    for ts in ("TS1", "TS2"):
        for d in dates:
            rows.append({
                "MMM_TIMESERIES_ID": ts,
                "ORGANISATION_ID": "ORG1",
                "DATE_DAY": d.strftime("%Y-%m-%d"),
                "CURRENCY_CODE": "USD",
                "ALL_PURCHASES": 100,
                "GOOGLE_PAID_SEARCH_SPEND": 10.0,
                "GOOGLE_SHOPPING_SPEND": 5.0,
                "GOOGLE_PMAX_SPEND": 2.0,
                "META_FACEBOOK_SPEND": 8.0,
                "META_INSTAGRAM_SPEND": 4.0,
            })
    raw = ROOT / "data" / "raw" / "uploaded_dataset.csv"
    raw.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(raw, index=False)
    df = run_pipeline(raw_path=str(raw.resolve()))["ready_df"]

eda = generate_eda_report(df)
eda["spend_over_time"]

FileNotFoundError: [Errno 2] No such file or directory: 'config.yaml'